# Spatial comparisons
---  
*J. Michelle Hu  
University of Utah  
Updated Nov 2024*  


In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
import geopandas as gpd
import xarray as xr
# import hvplot.xarray

from s3fs import S3FileSystem, S3Map

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
%reload_ext autoreload
%autoreload 2

### Env setup

In [ ]:
from pathlib import PurePath
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem
pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

### Directories and global variables

In [ ]:
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs/'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'
poly_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/polys'
aso_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO'

# SNOTEL all sites geojson fn - snotel site json
allsites_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/SNOTEL/snotel_sites_32613.json'

# nwm proj4 file
proj_fn = "/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/NWM_datasets_proj4.txt"

In [ ]:
# Basin-specific variables
# basin = 'blue'
basin = 'animas'
# basin = 'yampa'
WY = 2021

# Adjust basin dirs based on WY
basindirs = h.fn_list(workdir, f'{basin}*/wy{WY}/{basin}*solar_albedo/')
_ = [print(b) for b in basindirs]

# Figure out filenames
poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]
print(poly_fn)

In [ ]:
def clean_axes(ax, ticksoff=True, labelsoff=True, gridon=True, fc='k', aspect='equal'):
    if ticksoff:
        ax.set_xticks([])
        ax.set_yticks([])
    if labelsoff:
        ax.set_xlabel('')
        ax.set_ylabel('')
    if fc is not None:
        ax.set_facecolor(fc)
    if gridon:
        ax.grid(True)
    ax.set_aspect(aspect)

# Pull ASO to determine dates for models:  
- iSnobal  
- UA  
- NWM 
  
Slice differences by elevation (resampled to comparison datasets (ASO, UA, NWM)

In [ ]:
state = 'CO'
basinname = basin.capitalize()
endings = ['depth', 'swe']
inputvars = ['_snowdepth', '_swe']

# basinname = 'USCOBR'
# ending = ''
# inputvar = '.'
basinname, endings, inputvars

In [ ]:
aso_var_list = []
for ending, inputvar in zip(endings, inputvars):
    # Water year collections should all be post January so this should work
    aso_depth_fns = h.fn_list(aso_dir, f'{state}/*{basinname}*{WY}*{ending}*tif')

    # Load depth arrays and squeeze out single dimensions
    aso_depth_list = [np.squeeze(xr.open_dataset(fn)) for fn in aso_depth_fns]

    # Rename band_data to descriptive snow_depth
    aso_depth_list = [ds.rename_vars({'band_data': ending}) for ds in aso_depth_list]

    # Deal with adding time input for ASO data
    aso_depth_list = [np.squeeze(proc.assign_dt(ds, proc.extract_dt(fn, inputvar=inputvar))) for ds, fn in zip(aso_depth_list, aso_depth_fns)]

    _ = [print(f) for f in aso_depth_fns]

    aso_var_list.append(aso_depth_list)

# compute density and write out
density_list = []
for aso_template_fn, depth, swe in zip(aso_depth_fns, aso_var_list[0], aso_var_list[1]):
    density_fn = f"{aso_template_fn.split('swe')[0]}density{aso_template_fn.split('swe')[1]}"
    if not os.path.exists(density_fn):
        print('Computing density')
        density = swe['swe'] / depth['depth'] * 1000
        density_ds = xr.Dataset({'density': density})
        density_ds.attrs['units'] = 'kg/m^3'
        density_ds.attrs['long_name'] = 'Snow Density'
        density_ds.attrs['description'] = 'Snow density calculated from ASO SWE and ASO snow depth'
        density_ds.rio.set_spatial_dims('x', 'y', inplace=True)
        density_ds.rio.write_crs(depth.rio.crs, inplace=True)
        density_ds.rio.to_raster(density_fn)
    else:
        density_ds = xr.open_dataset(density_fn)
        density_ds = density_ds.rename_vars({'band_data': 'density'})
        density_ds = np.squeeze(proc.assign_dt(density_ds, proc.extract_dt(aso_template_fn, inputvar='_swe')))
    density_list.append(density_ds)

In [ ]:
cmap = 'Blues'
titles = [pd.to_datetime(depth.time.values).strftime('%Y-%m-%d') for depth in aso_depth_list]
cbaron = [False, True]
figsize = (12,4)
vmin = 0
vmax = 3

# Plot the snow depths
fig, axa = plt.subplots(1, 2, sharex=True, sharey=True, figsize=figsize)
for jdx, depth in enumerate(aso_depth_list):
    depth['swe'].plot.imshow(ax=axa[jdx], cmap=cmap, add_colorbar=cbaron[jdx], vmin=vmin, vmax=vmax)
    ax = axa[jdx]
    clean_axes(ax)
    # ax.set_aspect('equal')
    ax.set_title(titles[jdx])
    # ax.set_facecolor('black')

plt.tight_layout()

In [ ]:
# Plot up density
cmap = 'Blues'
titles = [pd.to_datetime(depth.time.values).strftime('%Y-%m-%d') for depth in aso_depth_list]
cbaron = [False, True]
figsize = (12,4)
vmin = 0
vmax = 1000

# Plot the ASO snow density
fig, axa = plt.subplots(1, 2, sharex=True, sharey=True, figsize=figsize)
for jdx, ds in enumerate(density_list):
    ds['density'].plot.imshow(ax=axa[jdx], cmap=cmap, add_colorbar=cbaron[jdx], vmin=vmin, vmax=vmax)
    ax = axa[jdx]
    clean_axes(ax)
    # ax.set_aspect('equal')
    ax.set_title(titles[jdx])
    # ax.set_facecolor('black')

plt.tight_layout()

## iSnobal bits

In [ ]:
# Get dates, could easily just pull from filenames, but this is fine
date_list = [proc.extract_dt(fn, inputvar=inputvar)[0] for fn in aso_depth_fns]
date_list = [f.strftime('%Y-%m-%d') for f in date_list]

In [ ]:
# Based on validation timesteps, select a time slice
ddx = 1
dt = date_list[ddx]

# convert to datetime object
dt = np.datetime64(dt)

In [ ]:
# find isnobal output and compare directly
isnobal_rundirs = [h.fn_list(basindirs[0], f"run{''.join(str(dt).split('-'))}")[0] for dt in date_list]
print(isnobal_rundirs)

drop_var_list = ['specific_mass', 'liquid_water', 'temp_surf', 'temp_lower', 'temp_snowcover', 'thickness_lower', 'water_saturation', 'projection']

# Read in snow depths and plot
depth_fns = [proc.fn_list(sdir, "snow.nc")[0] for sdir in isnobal_rundirs]
print(depth_fns)

# Open snow depth files
depths = [np.squeeze(xr.open_mfdataset(depth_fn, decode_coords="all", drop_variables=drop_var_list)) for depth_fn in depth_fns]

# Set CRS for iSnobal output
depths = [depth.rio.write_crs('epsg:32613', inplace=True) for depth in depths]


In [ ]:
# Compute SWE from datasets
for ds in depths:
    ds['swe'] = ds['thickness'] * ds['snow_density'] / 1000

In [ ]:
cmap = 'Blues'
titles = ['Baseline', 'HRRR-SPIReS']
cbaron = [False, True]
figsize = (12,3)
vmin, vmax = 0, 3
# Plot the iSnobal swe
fig, axa = plt.subplots(1, 2, sharex=True, sharey=True, figsize=figsize)
for jdx, depth in enumerate(depths):
    depth['swe'].plot.imshow(ax=axa[jdx], cmap=cmap, add_colorbar=cbaron[jdx], vmin=vmin, vmax=vmax)
    ax = axa[jdx]
    ax.set_aspect('equal')
    ax.set_title(titles[jdx])

plt.tight_layout()

## NWM

In [ ]:
import copy

In [ ]:
if WY > 2022:
    pass
else:
    bucket = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr'
    fs = S3FileSystem(anon=True)
    ds = xr.open_dataset(S3Map(f"{bucket}/ldasout.zarr", s3=fs), engine='zarr')

    # Read in NWM proj4 string
    with open(proj_fn, "r") as f:
        proj4 = f.read()
    print(proj4)

    # Read in poly_fn for spatial slicing
    epsg = 'epsg:32613'
    poly_gdf = gpd.read_file(poly_fn)
    poly_gdf.set_crs(epsg, inplace=True, allow_override=True)

    # Convert to NWM proj4 coords
    poly_gdf = poly_gdf.to_crs(crs=proj4)

    # Crop the dataset to the input polygon extent
    cropped_ds = ds.sel(x=slice(poly_gdf.bounds.minx.values[0], poly_gdf.bounds.maxx.values[0]),
                        y=slice(poly_gdf.bounds.miny.values[0], poly_gdf.bounds.maxy.values[0]))

    nwm_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/NWM/'
    nwm_swe_list = []
    for dt in date_list:
        # Slice time from NWM spatial crop
        nwm_depth = cropped_ds['SNOWH'].sel(time=dt)
        nwm_swe = cropped_ds['SNEQV'].sel(time=dt)

        # Establish CRS inplace
        nwm_depth = nwm_depth.rio.write_crs(input_crs=proj4, inplace=True)
        nwm_swe = nwm_swe.rio.write_crs(input_crs=proj4, inplace=True)

        # Extract x and y dims for density ds creation
        x, y = nwm_swe.x.values, nwm_swe.y.values

        # Compute density from daily mean SWE and daily mean snow depth
        nwm_density_data = nwm_swe.mean(dim='time').data / nwm_depth.mean(dim='time').data

        # Form the new xarray Dataset for density
        nwm_density = xr.Dataset(data_vars={
            "DENSITY": (("y", "x"), nwm_density_data),
            # "units": "kg m-3",
            # "description": "NWM daily mean snow density"
            },
            coords={
                "y": y,
                "x": x
                },
        )
        # Write this as the data variable name
        nwm_density.attrs['esri_pe_string'] = nwm_swe.attrs['esri_pe_string']
        # Note the units, longname, and description for the DENSITY data var
        nwm_density['DENSITY'].attrs['units'] = 'kg/m^3'
        nwm_density['DENSITY'].attrs['long_name'] = 'Snow Density'
        nwm_density['DENSITY'].attrs['description'] = 'Snow density calculated from mean daily NWM SNEQV and NWM SNOWH values'
        nwm_density.rio.write_crs(input_crs=proj4, inplace=True)

        # Write all out to file
        out_fn = f'{nwm_dir}/{basin}/{basin}_nwm_DENSITY_{dt}.nc'
        # if not os.path.exists(out_fn):
        print(f'Writing {out_fn}')
        for jdx, v in enumerate(['SNOWH', 'SNEQV', 'DENSITY']):
            out_fn = f'{nwm_dir}/{basin}/{basin}_nwm_{v}_{dt}.nc'
            if jdx == 0:
                nwm_depth.to_netcdf(out_fn)
            elif jdx == 1:
                nwm_swe.to_netcdf(out_fn)
            else:
                nwm_density.to_netcdf(out_fn)
        nwm_swe_list.append(nwm_swe)

In [ ]:
# Read it back in to check
nwm_density_fd = xr.open_dataset(out_fn)
nwm_density_fd['DENSITY'].plot.imshow()

## UA

In [ ]:
import fsspec

# Using the ASO dates and the basin iSnobal bounds,
cropped_list = []
for dt in date_list:
    # Convert to YYYYMMDD format
    dt = ''.join(dt.split('-'))
    # Query the UA data for the specific date (this includes both depth and SWE as variables)
    uris = f'https://climate.arizona.edu/data/UA_SWE/DailyData_800m/WY{WY}/UA_SWE_Depth_800m_v1_{dt}_stable.nc'

    # Prepend the cache type to the URIs, this is called protocol chaining in fsspec-speak
    file = fsspec.open_local(f"simplecache::{uris}", filecache={'cache_storage': '/tmp/fsspec_cache'})

    ua_ds = xr.open_mfdataset(file, engine="netcdf4", drop_variables=['time_str']) #h5netcdf not currently installed

    # Reassign coordinate names
    ua_ds = ua_ds.rename({'lat': 'y', 'lon': 'x'})

    # Explicitly write out the crs
    ua_ds.rio.write_crs(input_crs=ua_ds.crs.attrs['spatial_ref'], inplace=True)

    # Squeeze out the time dimension (single timestep)
    ua_ds = np.squeeze(ua_ds)

    # Convert depth and SWE to m (both are in millimeters)
    ua_ds['DEPTH'] = ua_ds['DEPTH'] / 1000
    ua_ds['SWE'] = ua_ds['SWE'] / 1000

    # Then compute density
    ua_ds['DENSITY'] = ua_ds['SWE'] / ua_ds['DEPTH'] * 1000


    # Note the units in the file
    ua_ds['DEPTH'].attrs['units'] = 'm'
    ua_ds['SWE'].attrs['units'] = 'm'
    ua_ds['DENSITY'].attrs['units'] = 'kg/m^3'

    # Write this to file in the ua_dir
    ancillarydir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products'
    ua_800m_dir = f'{ancillarydir}/UA/ua_800m/'
    if not os.path.exists(ua_800m_dir):
        os.makedirs(ua_800m_dir)
    outname = f'{ua_800m_dir}/UA_SWE_Depth_800m_v1_{dt}_stable_compiled.nc'

    # And save the UA data to netcdf
    ua_ds.to_netcdf(outname)

    # Read in poly_fn for spatial slicing
    epsg = 'epsg:32613'
    poly_gdf = gpd.read_file(poly_fn)
    poly_gdf.set_crs(epsg, inplace=True, allow_override=True)

    # Convert to UA crs
    poly_gdf = poly_gdf.to_crs(crs=ua_ds.crs.spatial_ref)
    # Then crop to iSnobal basin domain
    cropped_ds = ua_ds.sel(x=slice(poly_gdf.bounds.minx.values[0], poly_gdf.bounds.maxx.values[0]),
                           y=slice(poly_gdf.bounds.miny.values[0], poly_gdf.bounds.maxy.values[0]))
    cropped_ds = cropped_ds.rio.write_crs(input_crs=ua_ds.crs.spatial_ref, inplace=True)

    # And save that to file as well
    ua_basindir = f'{ua_800m_dir}/{basin}/'
    if not os.path.exists(ua_basindir):
        os.makedirs(ua_basindir)
    basin_outname = f'{ua_basindir}/UA_SWE_Depth_800m_v1_{dt}_stable_compiled_{basin}.nc'
    if not os.path.exists(basin_outname):
        print(f'Saving cropped dataset to {basin_outname}')
        cropped_ds.to_netcdf(basin_outname)
    cropped_list.append(cropped_ds)


In [ ]:
# read it back in
for dt in date_list:
    # Convert to YYYYMMDD format
    dt = ''.join(dt.split('-'))
    basin_outname = f'{ua_basindir}/UA_SWE_Depth_800m_v1_{dt}_stable_compiled_{basin}.nc'
    ua_fd = xr.open_dataset(basin_outname)
    fig, ax = plt.subplots(figsize=(4,2))
    ua_fd['DENSITY'].plot.imshow(ax=ax)
    h.clean_axes(ax)